# Shor algorithm profiling

Profiles the Ekera-Hastad version of Shor's algorithm for n=100 using:
* The Q# interpreter.
* `SequentialCountingProfiler`, which counts instructions and does nothing else.

This takes 28s.
The number of operations is $O(n^3)$, so the profiling time for n=2048 would be 28*(2048/100)^3 ~ 240K s.

Currently, we cannot emit QIR for this algorithm within a reasonable time because the loops are unrolled.
We know that profiling with QIR (using PyQIR and the bytecode interpreter) takes ~40 ns/op, which is ~50x faster. So, if we fix QIR generation, the profiling time for n=2048 would be 240000s/50=1.5 hours.

If we switch to the LLVM JIT with Inkwell, that would be ~4x faster, or ~20min.

To reproduce, you need to install the library containing the Q# implementation of this version of Shor's algorithm, which is currently in a private repository.

In [1]:
from time import perf_counter

from qdk.qre.profiler import QREProfiler

from shors.algorithms_qs import EkeraHastad2017Qs
from shors.utils import generate_factors

def report_result(result, t0):
    dt = perf_counter() - t0
    print("Logical qubits:", result.totals.logical_qubits)
    print("Total operations:", result.totals.total_operations_count)
    print("Operation counts:", result.totals.calls_per_operation)
    per_op = f"{int(1e9 * dt / result.totals.total_operations_count)} ns/op"
    print(f"Profiling time {dt:.2f}s, {per_op}")
    

BITLEN = 100
prime1, prime2 = generate_factors(BITLEN // 2, attempts=100)
composite = prime1 * prime2
algorithm = EkeraHastad2017Qs.create_default(composite)
app = algorithm.generate_circuit(base=2)

profiler = QREProfiler.counting()
t0 = perf_counter()
result = profiler.profile(app, use_qir=False)
report_result(result, t0)

Logical qubits: 457
Total operations: 12771632
Operation counts: {'cx': 5609255, 'rz': 2574, 'ccx': 1951700, 'z': 3957, 'h': 1007200, 'x': 1834188, 'mz_store': 3200, 'mz_store_reset': 1000763, 'read_result': 1003963, 'cz': 354732, 'reset': 100}
Profiling time 26.76s, 2095 ns/op
